## KinTree (Kinetic Interaction Tree)
#### KinTREE is a decision tree based model that is able to caputre kinetic interaction rules from molecular dynamics simulations.
##### This is an example notebook to compare between two systems.
Workflow Overview
1. Feature Extraction: Transform raw MD simulation trajectories into structured interaction data.
2. Model Training: Construct the KinTree to map the system A's kinetic hierarchy.
3. Load system B's data on the trained KinTree
4. Comparison: compare the difference in leaf occupancy between the two systems

### Step 1 Feature Extraction
This session preprocesses the data and prepares the input ready for the model. This includes:
1. **Sanity Checks**: Check whether the protontion states are correct, whether all residues have been defined by atom masks and stand amino acids list.
2. **Trajectory Alignment (Optional)**
3. **Trajectory Striding (Optional)**
2. **Calculation of the Contact map**
3. **Calculation of the Interaction network**
4. **Generate inputs**

In [1]:
import warnings
warnings.filterwarnings('ignore')
from Interaction_Detect import *
import MDAnalysis as mda
from MDAnalysis.analysis import align
from glob import glob
import torch

#### Critical Warning: Protonation States
Some molecular dynamics engines occasionally maintain standard residue names (e.g., **GLU**, **ASP**, or **CYS**) even after protonation changes. Since KinTree defines **atom masks** strictly based on these residue names, a mismatch will lead to incorrect interaction data.

Before proceeding, ensure your **protonation states align with your residue names.**

To facilitate this check, we provide the following checking function to validate your system. Things you should know for this function:
1. **Keep Hydrogens**: Your topology file must include hydrogen atoms
2. **Raw MD Output**: You can use the direct toplogy output from you simulation (including membrane components or solvent/salts). The function will filter for the relevant residues.

**Recommended Action**
Run the following cell to audit your topology. If the function flags a mismatch, you must either **update your residue names** in the .pdb/.gro file.

**Note that you might need to alter the function to suits your system**

In [2]:
def check_protonation_name_consistency(universe, selection="protein", verbose=True):
    """
    Check whether residue protonation state inferred from atom presence
    is consistent with residue name. Emits warnings (forced visible) and
    returns a list of mismatches.

    Parameters
    ----------
    universe : MDAnalysis.Universe
    selection : str
        Atom selection to scan (default "protein")
    verbose : bool
        Print a short summary + each mismatch line

    Returns
    -------
    mismatches : list[dict]
        Each dict contains segid, resid, resname, issue, details
    """
    ag = universe.select_atoms(selection)
    mismatches = []

    def record(res, issue, details):
        d = {
            "segid": str(res.segid),
            "resid": int(res.resid),
            "resname": str(res.resname),
            "issue": issue,
            "details": details,
        }
        mismatches.append(d)

    # Force warnings to show even in notebooks
    with warnings.catch_warnings():
        warnings.simplefilter("always", UserWarning)

        for res in ag.residues:
            resname = res.resname.strip()
            atom_names = {a.name.strip() for a in res.atoms}

            # ---- ASP / ASH: detect sidechain acidic H as HD1/HD2 (ignore HB*, HA, etc.) ----
            if resname in {"ASP", "ASH"}:
                has_sidechain_HD = any(n in atom_names for n in ("HD1", "HD2"))

                if resname == "ASP" and has_sidechain_HD:
                    msg = f"ASP has sidechain proton {sorted(set(atom_names)&{'HD1','HD2'})} but resname is ASP (expected ASH)"
                    warnings.warn(f"{msg} @ segid={res.segid}, resid={res.resid}", UserWarning)
                    record(res, "ASP_named_but_protonated", msg)

                if resname == "ASH" and not has_sidechain_HD:
                    msg = "ASH has no sidechain proton (HD1/HD2 missing) but resname is ASH (expected ASP)"
                    warnings.warn(f"{msg} @ segid={res.segid}, resid={res.resid}", UserWarning)
                    record(res, "ASH_named_but_deprotonated", msg)

            # ---- GLU / GLH: detect sidechain acidic H as HE1/HE2 ----
            if resname in {"GLU", "GLH"}:
                has_sidechain_HE = any(n in atom_names for n in ("HE1", "HE2"))

                if resname == "GLU" and has_sidechain_HE:
                    msg = f"GLU has sidechain proton {sorted(set(atom_names)&{'HE1','HE2'})} but resname is GLU (expected GLH)"
                    warnings.warn(f"{msg} @ segid={res.segid}, resid={res.resid}", UserWarning)
                    record(res, "GLU_named_but_protonated", msg)

                if resname == "GLH" and not has_sidechain_HE:
                    msg = "GLH has no sidechain proton (HE1/HE2 missing) but resname is GLH (expected GLU)"
                    warnings.warn(f"{msg} @ segid={res.segid}, resid={res.resid}", UserWarning)
                    record(res, "GLH_named_but_deprotonated", msg)

            # ---- Histidine: check tautomer consistency when named (skip plain HIS) ----
            if resname in {"HID", "HSD", "HIE", "HSE", "HIP", "HSP"}:
                has_HD1 = "HD1" in atom_names
                has_HE2 = "HE2" in atom_names

                expected = {
                    "HID": (True,  False),
                    "HSD": (True,  False),
                    "HIE": (False, True ),
                    "HSE": (False, True ),
                    "HIP": (True,  True ),
                    "HSP": (True,  True ),
                }[resname]

                found = (has_HD1, has_HE2)
                if found != expected:
                    msg = f"{resname} expected (HD1,HE2)={expected} but found {found}"
                    warnings.warn(f"{msg} @ segid={res.segid}, resid={res.resid}", UserWarning)
                    record(res, "HIS_tautomer_mismatch", msg)

            # ---- CYS / CYM: thiol proton HG ----
            # ---------- CYS / CYM / CYX ----------
            # ---------- CYS / CYM / CYX ----------
            if resname in {"CYS", "CYM", "CYX"}:
                has_thiol_H = any(a.name.startswith("HG") for a in res.atoms)

                if resname == "CYS" and not has_thiol_H:
                    msg = "CYS without thiol proton (may be CYM or CYX)"
                    warnings.warn(
                        f"{msg} (segid={res.segid}, resid={res.resid})",
                        UserWarning
                    )
                    record(res, "CYS_named_but_deprotonated", msg)

                if resname == "CYM" and has_thiol_H:
                    msg = "CYM with thiol proton (should be CYS)"
                    warnings.warn(
                        f"{msg} (segid={res.segid}, resid={res.resid})",
                        UserWarning
                    )
                    record(res, "CYM_named_but_protonated", msg)

                if resname == "CYX" and has_thiol_H:
                    msg = "CYX with thiol proton (disulfide cysteine should not have HG)"
                    warnings.warn(
                        f"{msg} (segid={res.segid}, resid={res.resid})",
                        UserWarning
                    )
                    record(res, "CYX_named_but_protonated", msg)



    if verbose:
        if mismatches:
            print(f"[protonation-name check] mismatches: {len(mismatches)}")
            for m in mismatches:
                print(f"  {m['segid']}:{m['resid']} {m['resname']} | {m['issue']} | {m['details']}")
        else:
            print("[protonation-name check] no mismatches found.")

    return mismatches

In [3]:
### example usage
u = mda.Universe('./data/x.pdb')
check_protonation_name_consistency(u)

FileNotFoundError: [Errno 2] No such file or directory: './data/x.pdb'

**Handling Protonation Mismatches**
If you see any mismatch in the previous cell, you must manually correct your topology with the expectd namings. Failure to do so will result in an inaccurate interaction detection.

**Step-by-Step Correction**
1. **Identify the Residue**: Note the residue ID and atom names flagged by the function. The residue ID provided by the function is the actualy ID in the pdb.
2. **Edit the Topology**: Open your topology file (e.g., .pdb, .gro, or .psf) in a text editor.
3. **Rename the Residue**: Locate the specific residue and change its name to the kinetically correct version.
    Example: If you have a deprotonated Cysteine involved in a disulfide bond, change CYS to CYX.
4. **Re-Verify**: Save the file and run the checking function again to confirm the error is resolved.

Note: If you only have a topology without hydrogen, then you have no choice but trust current protonation names.

In [ ]:
u = mda.Universe('../data/g1_apo_clean.pdb')
check_protonation_name_consistency(u)

#### Optional: Multi-Trajectory Alignment
The core KinTree workflow is optimized for a single trajectory. If your dataset contains **multiple trajectories**, you must **align them to a common reference frame** to ensure spatial consistency across all interaction data.

The following utility aligns each of your trajectories to a specified reference structure (e.g., a starting crystal structure or a representative frame).

In [ ]:
# --- Step 1. Specify paths ---
# Use the updated topology from here
topology = './data/g1_apo.pdb'
# Detect all trajectory files (can include subfolders)
trajectory_files = sorted(glob('./data/*/*.xtc'))  # or './Data/*.xtc' if flat folder

print(f"Detected {len(trajectory_files)} trajectories:")

In [ ]:
# Define the Reference structure
reference = mda.Universe(topology)

# List of short trajectories
traj_files = trajectory_files  # Replace with your file names

# Initialize writer for the combined trajectory
n_atoms = reference.atoms.n_atoms
writer = mda.Writer("./Aligned.xtc", n_atoms)

traj_lengths = []
for traj in traj_files:
    # Load trajectory
    u = mda.Universe(topology, traj)
    traj_lengths.append(len(u.trajectory))

    # Align trajectory to the reference
    ref_atoms = reference.select_atoms("protein")  # Adjust selection if needed
    mobile_atoms = u.select_atoms("protein")

    # Create alignment object
    alignment = align.AlignTraj(u, reference, select="protein", in_memory=True)
    alignment.run()

    # Write aligned frames to the combined trajectory
    for ts in u.trajectory:
        writer.write(u.atoms)

# Close the writer
writer.close()

print(f"Lengths: {traj_lengths}")
print(f"Total frames: {sum(traj_lengths)}")

**Preserving Trajectory Boundaries**\
When concatenating multiple simulation replicas, the trajectory length (frame count) for each individual run is stored as a metadata reference. This prevents the model from incorrectly assuming a physical transition exists between the last frame of "Trajectory A" and the first frame of "Trajectory B."

In [ ]:
with open('./traj_lengths.pkl', 'wb') as fp:
    pickle.dump(traj_lengths, fp)

#### Topology Validation and Atom Masking
Before running the interaction analysis, we must verify that every residue in your topology is recognized by the KinTree atom masks. This step ensures no chemical interactions are missed due to naming mismatches.

In [ ]:
topology = './data/Y221H.pdb'
trajectory = './data/Y221H.xtc'

In [ ]:
validate_topology(topology,atom_masks)

If the function returns a list of "undefined" residues, do not be alarmed.\
    - **Engine-Specific Naming**: KinTree’s default masks use AMBER nomenclature.\
    - **CHARMM/GROMOS Support**: If you are using a different force field, residues like Histidine might be named differently (e.g., HSD instead of HID).\
You can map unknown residue names to existing definitions without rewriting the underlying logic. This "aliasing" ensures the engine treats them identically.

In [ ]:
atom_masks['HSD'] = atom_masks["HID"]
atom_masks['HSE'] = atom_masks["HIE"]

You can save the atom mask for next time usage

In [ ]:
save_masks(atom_masks)

In [ ]:
atom_masks = load_masks('./atom_masks.json')

In [ ]:
atom_masks

In [ ]:
for res in list(atom_masks.keys()):
    # Iterate over copy of keys so we can delete while iterating
    for category in list(atom_masks[res].keys()):
        if not atom_masks[res][category]:  # Checks if list is empty []
            del atom_masks[res][category]  # Remove the key entirely

print("Atom masks cleaned: Empty categories removed.")

After mapping your force-field-specific residues, you must add them to the STANDARD_AMINO_ACIDS list.

This step is critical because the KinTree engine uses this list to identify protein backbone atoms. By default, the model filters out backbone-backbone interactions, as these are often ubiquitous and less "kinetically sensitive" than the side-chain interactions that drive specific conformational transitions.

In [ ]:
# Check the current standard amino acid
STANDARD_AMINO_ACIDS

In [ ]:
# Update if you have defined any amino acid names (like HSD)
# Just add to the set directly
STANDARD_AMINO_ACIDS.add("HSD")
STANDARD_AMINO_ACIDS.add("HSE")
STANDARD_AMINO_ACIDS

**Essential! Check again moving forward**

In [ ]:
validate_topology(topology,atom_masks)

#### Optional: Striding Long Trajectories
For systems with high atom counts or long simulation times, striding the trajectory can significantly accelerate the interaction detection and model training phases without losing the essential kinetic landscape.

In [ ]:
import MDAnalysis as mda
import pickle

# --- Inputs ---
topology = topology
trajectory = "./Aligned.xtc"
stride = 5

# --- Load original trajectory lengths ---
with open('./traj_lengths.pkl', 'rb') as fp:
    traj_lengths = pickle.load(fp)

# --- Create universe and writer ---
u = mda.Universe(topology, trajectory)

# --- Write strided trajectory ---
with mda.Writer("./Strided_MT1_prot.xtc", u.atoms.n_atoms) as W:
    for ts in u.trajectory[::stride]:
        W.write(u.atoms)

# --- Save new traj_lengths (just integer division) ---
traj_lengths_strided = [L // stride for L in traj_lengths]

with open("./traj_lengths_strided.pkl", "wb") as f:
    pickle.dump(traj_lengths_strided, f)

print("✅ Saved strided trajectory and 'traj_lengths_strided.pkl'")

In [ ]:
# --- Load current trajectory lengths ---
with open('./traj_lengths_strided.pkl', 'rb') as fp:
    traj_lengths = pickle.load(fp)

#### Calculation of Contact map and Interaction network
This session converts raw coordinates into a kinetic feature set by mapping residue-to-residue contacts.

1. **Benchmarking**: Determine the maximum batch size your hardware (GPU/RAM) can handle to optimize processing speed.

2. **Soft Contact Map**: Instead of binary cutoffs, we assign a continuous soft score to each pair. This captures subtle conformational fluctuations that rigid thresholds miss.

3. **Interaction Network**: Construct a weighted network using these scores. The previously defined atom_masks and STANDARD_AMINO_ACIDS are applied here to filter out backbone noise.

4. **Model Input**: Flatten the network tensors into a structured format.

In [ ]:
import pickle

def load_all_filtered_pairs(pkl_path):
    filtered_pairs_all = []

    with open(pkl_path, "rb") as f:
        while True:
            try:
                record = pickle.load(f)
            except EOFError:
                break

            # record["pairs_bin"] is a list of per-frame arrays
            filtered_pairs_all.extend(record["pairs_bin"])

    return filtered_pairs_all

##### Input Generation System 1

In [ ]:
# Speicfy the path to your topology file and trajectory file
trajectory = "./data/Aligned.xtc"
u = mda.Universe(topology, trajectory)
n_frames = len(u.trajectory)

if n_frames != sum(traj_lengths):
    print(f"ERROR: Total frames in Aligned.xtc ({n_frames}) does not match "
          f"sum of individual traj lengths ({sum(traj_lengths)})!")
else:
    print(f"Frame count OK: {n_frames} frames total.")

In [ ]:
# Test the optimal batch size based on your resources
optimal_batch = find_max_batch_size(topology, trajectory, max_frames=n_frames)
print(f"Using batch size: {optimal_batch}")

In [ ]:
# Conduct the contact_map for the whole protein
# 40min for a 60000 frame trajecotry of a protein with 450 residues on AMD 5000 Series CPU
# --- Step 3. Compute contacts for all trajectories ---
# if you want to consider any water or ions. you need to specify their name in ligand_names variable!
os.makedirs('./data', exist_ok=True)
filtered_pairs_all = contact_map(
    topology=topology,
    trajectory=trajectory,
    batch_size=optimal_batch,
    outputpath='./data/contact_map_softcontacts.pkl'
)

print("\n✅ Contact map calculation complete.")
print("Trajectory lengths:", traj_lengths)
print("Total frames:", sum(traj_lengths))

In [ ]:
# Detect interactions based on the calculated contact map
# The customized thresholds is a txt file storing all the user defined threshold

# Define the thresholds content
# Parameters available for user design are:
# hydrogen bond distance threshold
# salt bridge distance threshold
# pi-pi interaction distance, angle, and slippage
# T shape interaction angle (distance and slippage are identical to pi-pi)
# cation-pi interaction distance and angle

# A sample is as follow, with default settings

thresholds_content = """# Interaction thresholds
hbond: 4.0
salt_bridge: 4.0
pi_pi: 6.0, 30.0, 3.5
T_shape: 30.0
cation_pi: 6.0, 30.0
"""

saving_path = "interaction_threshold.txt"

# Write to a txt file
with open(saving_path, "w") as f:
    f.write(thresholds_content)

print("interaction threshold file has been created!")

In [ ]:
with open('./data/contact_map_softcontacts.pkl', 'rb') as f:
    data = pickle.load(f)

_contact_map = data

print(f"Loaded {len(_contact_map)} frames from saved file.")
print(f"Trajectory lengths: {traj_lengths}")

In [ ]:
filtered_pairs_all = load_all_filtered_pairs(
    "./data/contact_map_softcontacts.pkl"
)

print(f"Loaded {len(filtered_pairs_all)} frames from saved file.")

In [ ]:
# Calculte the interaction from the calculated contact map
# at least 2h for a 60000 frame trajecotry of a protein with 450 residues on AMD 5000 Series CPU
interaction_detect_ = interaction_detect(topology, trajectory, filtered_pairs_all, customized_thresholds=saving_path,
                                         atom_masks=atom_masks,outputpath='./data/interaction.pkl')

print("\n✅ Interaction detection complete.")
print("Trajectory lengths:", traj_lengths)
print("Total frames:", sum(traj_lengths))

In [ ]:
# This would generate input without contacts
# One should see which works better
generate_interaction_input(
    interaction_detect_path="./data/interaction.pkl",
    outputpath="./data/interaction.pt"
)

In [ ]:
# Check the size of the save PyG file again
# Load the stored PyG file
data_dict = torch.load("./data/interaction.pt")

edge_index = data_dict["edge_index"]          # shape (2, N_edges)
edge_attr_all = data_dict["edge_attr_all"]    # list of [N_edges, F] tensors
interaction_types = data_dict["interaction_types"]

print("edge_index shape:", edge_index.shape)              # (2, N_edges)
print("num frames:", len(edge_attr_all))
print("edge_attr shape (first frame):", edge_attr_all[0].shape)  # (N_edges, F)
print("interaction types:", interaction_types)

##### Input generation system 2

In [ ]:
# Speicfy the path to your topology file and trajectory file
trajectory = "./data/Aligned.xtc"
u = mda.Universe(topology, trajectory)
n_frames = len(u.trajectory)

In [ ]:
if n_frames != sum(traj_lengths):
    print(f"ERROR: Total frames in Aligned.xtc ({n_frames}) does not match "
          f"sum of individual traj lengths ({sum(traj_lengths)})!")
else:
    print(f"Frame count OK: {n_frames} frames total.")

In [ ]:
# Test the optimal batch size based on your resources
optimal_batch = find_max_batch_size(topology, trajectory, max_frames=n_frames)
print(f"Using batch size: {optimal_batch}")

In [ ]:
# Conduct the contact_map for the whole protein
# 40min for a 60000 frame trajecotry of a protein with 450 residues on AMD 5000 Series CPU
# --- Step 3. Compute contacts for all trajectories ---
# if you want to consider any water or ions. you need to specify their name in ligand_names variable!
os.makedirs('./data', exist_ok=True)
filtered_pairs_all = contact_map(
    topology=topology,
    trajectory=trajectory,
    batch_size=optimal_batch,
    outputpath='./data/contact_map_softcontacts.pkl'
)

print("\n✅ Contact map calculation complete.")
print("Trajectory lengths:", traj_lengths)
print("Total frames:", sum(traj_lengths))

In [ ]:
# Detect interactions based on the calculated contact map
# The customized thresholds is a txt file storing all the user defined threshold

# Define the thresholds content
# Parameters available for user design are:
# hydrogen bond distance threshold
# salt bridge distance threshold
# pi-pi interaction distance, angle, and slippage
# T shape interaction angle (distance and slippage are identical to pi-pi)
# cation-pi interaction distance and angle

# A sample is as follow, with default settings

thresholds_content = """# Interaction thresholds
hbond: 4.0
salt_bridge: 4.0
pi_pi: 6.0, 30.0, 3.5
T_shape: 30.0
cation_pi: 6.0, 30.0
"""

saving_path = "interaction_threshold.txt"

# Write to a txt file
with open(saving_path, "w") as f:
    f.write(thresholds_content)

print("interaction threshold file has been created!")

In [ ]:
with open('./data/contact_map_softcontacts.pkl', 'rb') as f:
    data = pickle.load(f)

_contact_map = data

print(f"Loaded {len(_contact_map)} frames from saved file.")
print(f"Trajectory lengths: {traj_lengths}")

In [ ]:
filtered_pairs_all = load_all_filtered_pairs(
    "./data/contact_map_softcontacts.pkl"
)

print(f"Loaded {len(filtered_pairs_all)} frames from saved file.")

In [ ]:
# Calculte the interaction from the calculated contact map
# at least 2h for a 60000 frame trajecotry of a protein with 450 residues on AMD 5000 Series CPU
interaction_detect_ = interaction_detect(topology, trajectory, filtered_pairs_all, customized_thresholds=saving_path,
                                         atom_masks=atom_masks, outputpath='./data/interaction.pkl')

print("\n✅ Interaction detection complete.")
print("Trajectory lengths:", traj_lengths)
print("Total frames:", sum(traj_lengths))

In [ ]:
# This would generate input without contacts
# One should see which works better
generate_interaction_input(
    interaction_detect_path="./data/interaction.pkl",
    outputpath="./data/interaction.pt"
)

In [ ]:
# Check the size of the save PyG file again
# Load the stored PyG file
data_dict = torch.load("./data/interaction.pt")

edge_index = data_dict["edge_index"]  # shape (2, N_edges)
edge_attr_all = data_dict["edge_attr_all"]  # list of [N_edges, F] tensors
interaction_types = data_dict["interaction_types"]

print("edge_index shape:", edge_index.shape)  # (2, N_edges)
print("num frames:", len(edge_attr_all))
print("edge_attr shape (first frame):", edge_attr_all[0].shape)  # (N_edges, F)
print("interaction types:", interaction_types)

### Step 2: Model Training To Compare the Two systems
This session identifies the kinetically essential interactions that drive your system's transitions.

1. **Apply Hysteresis**: We use a dual-threshold filter to remove high-frequency "flickering" noise from transient contacts, ensuring only stable transitions are modeled.

2. **Filter Contact Map**: The soft contact map is refined to focus on essential residues, to reduce computation complexity and provide more detailed structural information

3. **Model Training**: The KinTree algorithm builds a hierarchical map of metastable states based on these filtered interaction networks.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.decomposition import PCA
import random
from temporal_rule_tree import *
from msm_kinetics import *

In [ ]:
# Force deterministic behavior
seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

In [ ]:
# ----------------------------
# paths
# ----------------------------
WT_contact_pkl   = "./data/contact_map_WT.pkl"
WT_inter_pkl  = "./data/interaction_WT.pkl"

MT_contact_pkl   = "./data/contact_map_MT.pkl"
MT_inter_pkl  = "./data/interaction_MT.pkl"

WT_shared_pt   = "./data/WT_shared_no_contact.pt"
MT_shared_pt = "./data/MT_shared_no_contact.pt"

# IMPORTANT: no "contact" here
INTERACTION_TYPES = ['hbond', 'salt_bridge', 'pi_pi', 'T-shape', 'cation_pi']

In [ ]:
# ----------------------------
# loaders
# ----------------------------
def load_pickle_any(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def load_interaction_frames(interaction_detect_path):
    data = load_pickle_any(interaction_detect_path)

    if isinstance(data, dict) and "interactions_all" in data:
        data = data["interactions_all"]

    if not isinstance(data, list):
        raise ValueError(f"Unexpected interaction format in {interaction_detect_path}: {type(data)}")

    return data


# reconstruct full trajectory from chunked soft-contact file
def load_binary_contact_frames_from_soft(contact_path):
    frames = load_all_filtered_pairs(contact_path)   # your existing reconstruction function

    cleaned = []
    for frame in frames:
        arr = np.asarray(frame)
        frame_pairs = []

        if arr.size == 0:
            cleaned.append(frame_pairs)
            continue

        if arr.ndim != 2 or arr.shape[1] < 2:
            raise ValueError(f"Bad frame shape in {contact_path}: {arr.shape}")

        for row in arr:
            frame_pairs.append((int(row[0]), int(row[1])))

        cleaned.append(frame_pairs)

    return cleaned

In [ ]:
# ----------------------------
# edge collection
# only from NON-CONTACT interactions
# ----------------------------
def collect_edges_from_interactions_only(interaction_frames, interaction_types=INTERACTION_TYPES):
    edge_set = set()

    for frame_dict in interaction_frames:
        for kind in interaction_types:
            for item in frame_dict.get(kind, []):
                if isinstance(item, tuple) and len(item) == 2 and isinstance(item[0], tuple):
                    (i, j), _ = item
                else:
                    i, j = item
                edge_set.add(tuple(sorted((int(i), int(j)))))

    return edge_set

In [ ]:
# ----------------------------
# shared pt builder
# only NON-CONTACT channels
# ----------------------------
def generate_interaction_input_with_shared_edges(interaction_frames, outputpath, global_edges,
                                                interaction_types=INTERACTION_TYPES):
    interaction_to_index = {k: i for i, k in enumerate(interaction_types)}
    edge_to_index = {pair: idx for idx, pair in enumerate(global_edges)}
    edge_index = torch.tensor(global_edges, dtype=torch.long).t().contiguous()

    edge_attr_all = []
    for frame_idx in range(len(interaction_frames)):
        edge_attr = torch.zeros(len(global_edges), len(interaction_types), dtype=torch.uint8)

        for kind, pairs in interaction_frames[frame_idx].items():
            if kind not in interaction_to_index:
                continue

            for item in pairs:
                if isinstance(item, tuple) and len(item) == 2 and isinstance(item[0], tuple):
                    (i, j), _ = item
                else:
                    i, j = item

                pair = tuple(sorted((int(i), int(j))))
                idx = edge_to_index.get(pair)
                if idx is not None:
                    edge_attr[idx, interaction_to_index[kind]] = 1

        edge_attr_all.append(edge_attr)

    torch.save({
        "edge_index": edge_index,
        "edge_attr_all": edge_attr_all,
        "interaction_types": interaction_types
    }, outputpath)

    print(f"Saved: {outputpath}")
    print(f"  frames = {len(edge_attr_all)}")
    print(f"  edges  = {edge_index.shape[1]}")
    print(f"  types  = {interaction_types}")

In [ ]:
# ----------------------------
# load interaction data
# ----------------------------
WT_inter_frames = load_interaction_frames(WT_inter_pkl)
MT_inter_frames = load_interaction_frames(MT_inter_pkl)

print("System 1 interaction frames:", len(WT_inter_frames))
print("System 2 interaction frames:", len(MT_inter_frames))

In [ ]:
# ----------------------------
# build shared NON-CONTACT edge space
# ----------------------------
edges_prot = collect_edges_from_interactions_only(WT_inter_frames)
edges_deprot = collect_edges_from_interactions_only(MT_inter_frames)
shared_global_edges = sorted(edges_prot | edges_deprot)

print("System 1 unique interaction edges   :", len(edges_prot))
print("System 2 unique interaction edges :", len(edges_deprot))
print("shared unique interaction edges :", len(shared_global_edges))

In [ ]:
# ----------------------------
# save shared .pt
# ----------------------------
generate_interaction_input_with_shared_edges(
    WT_inter_frames, WT_shared_pt, shared_global_edges
)

generate_interaction_input_with_shared_edges(
    MT_inter_frames, MT_shared_pt, shared_global_edges
)

In [ ]:
# ----------------------------
# contact_frames for the tree
# kept separately; NOT part of X features
# ----------------------------
filtered_pairs_all = load_all_filtered_pairs(WT_contact_pkl)
contact_frames_prot = [
    [(int(row[0]), int(row[1]), 1.0) for row in np.asarray(frame)]
    for frame in filtered_pairs_all
]

print("tree contact_frames:", len(contact_frames_prot))
print("first tree contact entry:", contact_frames_prot[0][0] if len(contact_frames_prot[0]) else "empty")

In [ ]:
# Model training preparation
X_WT, feature_names_WT, feature_types_WT = load_interaction_binary_pt(
    "./data/WT_shared_no_contact.pt",
    use_types_separately=True,
    present_threshold=0.5
)

# Model training preparation
X_MT, feature_names_MT, feature_types_MT = load_interaction_binary_pt(
    "./data/MT_shared_no_contact.pt",
    use_types_separately=True,
    present_threshold=0.5
)
#contacts = load_contact_map_pkl('./data/contact_map_softcontacts.pkl')

In [ ]:
traj_lengths=[30000]

In [ ]:
min_on, min_off, on_lengths, off_lengths = suggest_global_hysteresis_thresholds(
    X_WT, traj_lengths=traj_lengths, on_percentile=95, off_percentile=95
)

min_on, min_off, on_lengths, off_lengths = suggest_global_hysteresis_thresholds(
    X_MT, traj_lengths=traj_lengths, on_percentile=95, off_percentile=95
)

In [ ]:
# --- τ prescan from debounced features (PCA + KMeans) ---
# 0) Debounce first (match the tree’s input)
# Increase min_on and min_off if the following test cannot pass the ck test
WT_hysteresis = apply_hysteresis(X_WT, min_on=16, min_off=16, traj_lengths=traj_lengths)

MT_hysteresis = apply_hysteresis(X_MT, min_on=17, min_off=17, traj_lengths=traj_lengths)

In [ ]:
# =========================
# main analysis
# =========================
T, F = X_WT.shape

# Suggested starting parameters
short_on_thr = 1
short_off_thr = 1
pct_modified_thr = 0.15
min_raw_support = 0.05
max_occupancy_shift = 0.10
safe_occupancy_shift = 0.02
safe_helpful_ratio = 0.90
max_long_run_damage = 0.05
common_feature_raw_frac = 0.20

results = []

for f in range(F):
    raw = X_WT[:, f].astype(int)
    smooth = WT_hysteresis[:, f].astype(int)

    row = classify_feature_action(
        raw=raw,
        smooth=smooth,
        T=T,
        short_on_thr=short_on_thr,
        short_off_thr=short_off_thr,
        pct_modified_thr=pct_modified_thr,
        min_raw_support=min_raw_support,
        max_occupancy_shift=max_occupancy_shift,
        safe_occupancy_shift=safe_occupancy_shift,
        safe_helpful_ratio=safe_helpful_ratio,
        max_long_run_damage=max_long_run_damage,
        common_feature_raw_frac=common_feature_raw_frac,
    )

    row["feature_idx"] = f
    row["feature_name"] = feature_names_WT[f]
    row["revert_old"] = row["pct_modified"] > pct_modified_thr
    results.append(row)

df = pd.DataFrame(results)

# =========================
# build final hybrid matrix
# =========================
X_hybrid_partial = WT_hysteresis.copy().astype(int)

n_keep = 0
n_partial = 0
n_full = 0

for f in range(F):
    raw = X_WT[:, f].astype(int)
    smooth = WT_hysteresis[:, f].astype(int)

    action = df.loc[f, "action_new"]

    if action == "keep_smoothed":
        n_keep += 1
        continue

    elif action == "full_revert":
        X_hybrid_partial[:, f] = raw
        n_full += 1

    elif action == "partial_revert":
        revert_mask = _make_partial_revert_mask(
            raw, smooth,
            short_on_thr=short_on_thr,
            short_off_thr=short_off_thr
        )
        X_hybrid_partial[revert_mask, f] = raw[revert_mask]
        n_partial += 1

    else:
        raise ValueError(f"Unknown action: {action}")

print("--- New hybrid summary ---")
print(f"keep_smoothed : {n_keep}")
print(f"partial_revert: {n_partial}")
print(f"full_revert   : {n_full}")

# =========================
# inspect the target feature
# =========================
target_interaction = "64-216:salt_bridge"
specific_result = df[df["feature_name"] == target_interaction]

print("\n--- Specific Interaction Result ---")
if not specific_result.empty:
    print(specific_result.to_string(index=False))
else:
    print(f"Interaction '{target_interaction}' not found.")

# =========================
# compare old vs new
# =========================
print("\n--- Action counts ---")
print(df["action_new"].value_counts())

print("\n--- Old revert but new partial/keep ---")
df_old_vs_new = df[df["revert_old"] == True].copy()
print(
    df_old_vs_new.sort_values("pct_modified", ascending=False)[
        [
            "feature_idx", "feature_name",
            "pct_modified", "raw_frac", "smooth_frac",
            "helpful_edit_ratio", "long_run_damage",
            "action_new", "reason_new"
        ]
    ].head(40).to_string(index=False)
)

In [ ]:
# Use this in downstream workflow
WT_hysteresis = X_hybrid_partial

In [ ]:
# =========================
# main analysis
# =========================
T, F = X_MT.shape

# Suggested starting parameters
short_on_thr = 1
short_off_thr = 1
pct_modified_thr = 0.15
min_raw_support = 0.05
max_occupancy_shift = 0.10
safe_occupancy_shift = 0.02
safe_helpful_ratio = 0.90
max_long_run_damage = 0.05
common_feature_raw_frac = 0.20

results = []

for f in range(F):
    raw = X_MT[:, f].astype(int)
    smooth = MT_hysteresis[:, f].astype(int)

    row = classify_feature_action(
        raw=raw,
        smooth=smooth,
        T=T,
        short_on_thr=short_on_thr,
        short_off_thr=short_off_thr,
        pct_modified_thr=pct_modified_thr,
        min_raw_support=min_raw_support,
        max_occupancy_shift=max_occupancy_shift,
        safe_occupancy_shift=safe_occupancy_shift,
        safe_helpful_ratio=safe_helpful_ratio,
        max_long_run_damage=max_long_run_damage,
        common_feature_raw_frac=common_feature_raw_frac,
    )

    row["feature_idx"] = f
    row["feature_name"] = feature_names_MT[f]
    row["revert_old"] = row["pct_modified"] > pct_modified_thr
    results.append(row)

df = pd.DataFrame(results)

# =========================
# build final hybrid matrix
# =========================
X_hybrid_partial = MT_hysteresis.copy().astype(int)

n_keep = 0
n_partial = 0
n_full = 0

for f in range(F):
    raw = X_MT[:, f].astype(int)
    smooth = MT_hysteresis[:, f].astype(int)

    action = df.loc[f, "action_new"]

    if action == "keep_smoothed":
        n_keep += 1
        continue

    elif action == "full_revert":
        X_hybrid_partial[:, f] = raw
        n_full += 1

    elif action == "partial_revert":
        revert_mask = _make_partial_revert_mask(
            raw, smooth,
            short_on_thr=short_on_thr,
            short_off_thr=short_off_thr
        )
        X_hybrid_partial[revert_mask, f] = raw[revert_mask]
        n_partial += 1

    else:
        raise ValueError(f"Unknown action: {action}")

print("--- New hybrid summary ---")
print(f"keep_smoothed : {n_keep}")
print(f"partial_revert: {n_partial}")
print(f"full_revert   : {n_full}")

# =========================
# inspect the target feature
# =========================
#target_interaction = "273-379:hbond"
# specific_result = df[df["feature_name"] == target_interaction]
#
# print("\n--- Specific Interaction Result ---")
# if not specific_result.empty:
#     print(specific_result.to_string(index=False))
# else:
#     print(f"Interaction '{target_interaction}' not found.")

# =========================
# compare old vs new
# =========================
print("\n--- Action counts ---")
print(df["action_new"].value_counts())

print("\n--- Old revert but new partial/keep ---")
df_old_vs_new = df[df["revert_old"] == True].copy()
print(
    df_old_vs_new.sort_values("pct_modified", ascending=False)[
        [
            "feature_idx", "feature_name",
            "pct_modified", "raw_frac", "smooth_frac",
            "helpful_edit_ratio", "long_run_damage",
            "action_new", "reason_new"
        ]
    ].head(40).to_string(index=False)
)

# Use this in downstream workflow
MT_hysteresis = X_hybrid_partial

In [ ]:
import torch
import numpy as np

prot_pt = "./data/WT_shared_no_contact.pt"
deprot_pt = "./data/MT_shared_no_contact.pt"

prot = torch.load(prot_pt, map_location="cpu")
deprot = torch.load(deprot_pt, map_location="cpu")

# ----------------------------
# 1. basic metadata checks
# ----------------------------
prot_edge_index = np.array(prot["edge_index"])
deprot_edge_index = np.array(deprot["edge_index"])

prot_types = list(prot["interaction_types"])
deprot_types = list(deprot["interaction_types"])

print("prot edge_index shape   :", prot_edge_index.shape)
print("deprot edge_index shape :", deprot_edge_index.shape)
print("prot interaction types  :", prot_types)
print("deprot interaction types:", deprot_types)

assert prot_edge_index.shape == deprot_edge_index.shape, "edge_index shapes differ"
assert prot_types == deprot_types, "interaction_types differ"

# ----------------------------
# 2. exact edge alignment check
# same column in edge_index must be same residue pair
# ----------------------------
same_edge_index = np.array_equal(prot_edge_index, deprot_edge_index)
print("\nExact edge_index identical:", same_edge_index)

if not same_edge_index:
    # print first few mismatches
    n_edges = min(prot_edge_index.shape[1], deprot_edge_index.shape[1])
    mismatches = []
    for k in range(n_edges):
        p = tuple(prot_edge_index[:, k].tolist())
        d = tuple(deprot_edge_index[:, k].tolist())
        if p != d:
            mismatches.append((k, p, d))
        if len(mismatches) >= 10:
            break

    print("First mismatches:")
    for k, p, d in mismatches:
        print(f"  column {k}: prot={p}, deprot={d}")
else:
    print("All edge columns are aligned.")

# ----------------------------
# 3. build human-readable feature names
# same flattened feature position should mean same edge:type
# ----------------------------
def make_feature_names(edge_index, interaction_types):
    edge_names = [f"{int(edge_index[0, e])}-{int(edge_index[1, e])}" for e in range(edge_index.shape[1])]
    feature_names = [f"{edge}:{itype}" for edge in edge_names for itype in interaction_types]
    return feature_names

prot_feature_names = make_feature_names(prot_edge_index, prot_types)
deprot_feature_names = make_feature_names(deprot_edge_index, deprot_types)

same_feature_names = (prot_feature_names == deprot_feature_names)
print("\nExact flattened feature alignment:", same_feature_names)

if not same_feature_names:
    for i, (a, b) in enumerate(zip(prot_feature_names, deprot_feature_names)):
        if a != b:
            print(f"First feature mismatch at position {i}:")
            print("  prot  :", a)
            print("  deprot:", b)
            break
else:
    print("All flattened feature positions are aligned.")

# ----------------------------
# 4. random spot checks
# verify several positions manually
# ----------------------------
rng = np.random.default_rng(0)
n_check = 10
rand_cols = rng.choice(prot_edge_index.shape[1], size=min(n_check, prot_edge_index.shape[1]), replace=False)

print("\nRandom edge column spot checks:")
for k in rand_cols:
    pair = tuple(prot_edge_index[:, k].tolist())
    print(f"  edge column {k}: pair={pair}")

rand_feats = rng.choice(len(prot_feature_names), size=min(n_check, len(prot_feature_names)), replace=False)
print("\nRandom flattened feature spot checks:")
for k in rand_feats:
    print(f"  feature column {k}: {prot_feature_names[k]}")

# ----------------------------
# 5. optional per-frame tensor shape checks
# ----------------------------
prot_edge_attr_all = prot["edge_attr_all"]
deprot_edge_attr_all = deprot["edge_attr_all"]

print("\nNumber of frames:")
print("  prot  :", len(prot_edge_attr_all))
print("  deprot:", len(deprot_edge_attr_all))

print("\nPer-frame tensor shape check:")
print("  prot frame 0 shape  :", tuple(prot_edge_attr_all[0].shape))
print("  deprot frame 0 shape:", tuple(deprot_edge_attr_all[0].shape))

assert tuple(prot_edge_attr_all[0].shape) == tuple(deprot_edge_attr_all[0].shape), \
    "Per-frame edge_attr shape differs between systems"

print("\nSanity check passed: shared .pt files are aligned in edge order and feature order.")

In [ ]:
import pickle
import numpy as np

def load_and_filter_contacts(pkl_path, important_residues):
    """
    Loads chunked contact data, filters for important residues,
    and returns a list of lists of tuples [(i, j, score), ...].
    """
    # Convert to numpy array for fast vectorized comparison
    important_arr = np.array(list(set(important_residues)))

    filtered_contacts_all = []
    total_frames = 0

    print(f"Reading and filtering from: {pkl_path}")

    with open(pkl_path, "rb") as f:
        while True:
            try:
                # Load one chunk record at a time
                record = pickle.load(f)
            except EOFError:
                # End of file reached
                break

            # Iterate through the frames in this chunk
            # record['scores'] is a list of np.arrays [(N,3), ...]
            for frame_arr in record['scores']:
                if frame_arr.size == 0:
                    filtered_contacts_all.append([])
                    continue

                # Columns: 0 = residue_i, 1 = residue_j, 2 = score
                # Cast indices to integer for comparison
                i_idx = frame_arr[:, 0].astype(int)
                j_idx = frame_arr[:, 1].astype(int)

                # Vectorized Filter: Check if i OR j is in important_arr
                # np.isin is much faster than list comprehension for large arrays
                mask = np.isin(i_idx, important_arr) | np.isin(j_idx, important_arr)

                # Apply the mask to keep only relevant rows
                filtered_data = frame_arr[mask]

                # Convert back to list of tuples [(i, j, s), ...]
                # to match your original data structure exactly
                frame_contacts = [
                    (int(row[0]), int(row[1]), float(row[2]))
                    for row in filtered_data
                ]

                filtered_contacts_all.append(frame_contacts)

            total_frames += record["n_frames"]
            print(f"Processed chunk {record['chunk_idx']+1} (Total frames: {total_frames})", end='\r')

    print(f"\nDone. Loaded {len(filtered_contacts_all)} frames.")
    return filtered_contacts_all

In [ ]:
# Your residues of interest
important_residues = [63, 64, 65, 66, 149, 150, 151, 314, 315, 316, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 118, 119, 313, 317, 318, 341, 342, 344, 345, 348]

input_file_WT = './data/contact_map_WT.pkl'
input_file_MT = './data/contact_map_MT.pkl'

# Run the loader
filtered_contacts_WT = load_and_filter_contacts(input_file_WT, important_residues)
filtered_contacts_MT = load_and_filter_contacts(input_file_MT, important_residues)

# Verify
if filtered_contacts_WT:
    print(f"Filtered contact count (frame 0): {len(filtered_contacts_WT[0])}")
    # Example output: [(21, 150, 0.85), (21, 151, 0.44), ...]
if filtered_contacts_MT:
    print(f"Filtered contact count (frame 0): {len(filtered_contacts_MT[0])}")

In [ ]:
with open("./data/filtered_contacts_WT.pkl", "wb") as f:
    pickle.dump(filtered_contacts_WT, f)
print("Saved filtered contact map as filtered_contacts.pkl")

with open("./data/filtered_contacts_MT.pkl", "wb") as f:
    pickle.dump(filtered_contacts_MT, f)
print("Saved filtered contact map as filtered_contacts.pkl")

In [ ]:
# Restore the data
with open("./data/filtered_contacts_WT.pkl", "rb") as f:
    filtered_contacts_WT = pickle.load(f)

with open("./data/filtered_contacts_MT.pkl", "rb") as f:
    filtered_contacts_MT = pickle.load(f)

print(f"Successfully loaded {len(filtered_contacts_WT)} frames.")
print(f"Successfully loaded {len(filtered_contacts_MT)} frames.")

In [ ]:
##### Train the WT tree

In [ ]:
tree = TemporalRuleTreeV6(
        lags=[5,6,7], # list of lag times
        alpha_vamp=0.5, # VAMP vs NMI weight
        min_pairs=2000, # minimum transition pairs per node
        min_leaf=2000, # minimum frames per leaf
        max_depth=5, # tree depth limit
        max_features_to_try=None, # try all features per split
        feature_types=feature_types_WT, # feature type labels
        random_state=seed, # RNG seed
        # kinetic-aware parameters
        lag_mode="uniform", # emphasize the largest lag (40–60 → 60)
        tau_power=1.0, # used if lag_mode="power"
        lambda_row=0.7, # penalty weight for kinetically-similar children
        contact_frames=filtered_contacts_WT,
        local_window=2,
        delta_mode='quantile',
        delta_quantile=0.6
        #delta_thresh=0.2
)

In [ ]:
from temporal_rule_tree import seqsep_soft_weights
w_feat = seqsep_soft_weights(
    feature_names_WT,
    min_sep=7,      # base threshold
    gamma=2.0,      # stronger penalty as sep gets smaller
    floor=0.15,     # never drop below 0.15 (to keep a faint chance)
    by_type={"hbond": 6, "salt_bridge": 5, "pi_pi": 8, "T_shape": 8, "cation_pi": 6}
)

In [ ]:
# Set weights for different interactions
tree.set_weights(feature_weights=w_feat,
                  type_weight_map={"hbond": 1.0,
                                   "salt_bridge": 1.5,
                                   "pi_pi": 0.7,
                                   "T_shape":0.5,
                                   "cation_pi":1.0})

In [ ]:
# Train the model
tree.fit(WT_hysteresis, feature_names=feature_names_WT, traj_lengths=traj_lengths)

In [ ]:
#### Compare between two systems

In [ ]:
# 3) project both systems into the protonated reference tree
leaf_WT = tree.predict_leaf_ids(WT_hysteresis)
leaf_MT = tree.predict_leaf_ids(MT_hysteresis)

print("n leaves:", int(max(leaf_prot.max(), leaf_deprot.max()) + 1))
print("WT leaf counts:", np.bincount(leaf_WT))
print("MT leaf counts:", np.bincount(leaf_MT, minlength=int(max(leaf_prot.max(), leaf_deprot.max()) + 1)))

In [ ]:
tree_dict = save_tree_json(tree, "results/WT.json")

In [ ]:
# Features used in tree
split_features = collect_split_features(tree_dict, tree)
print("Features driving splits:", split_features)

In [ ]:
import pandas as pd
import re

def renumber_pdb(pdb_input, excel_input, pdb_output):
    # 1. Load the numbering from Excel
    df = pd.read_excel(excel_input)
    # Assume Column 2 (index 1) contains the new indices
    raw_indices = df.iloc[:, 4].astype(str).tolist()

    # 2. Prepare the tetramer indices (Repeat the list 4 times)
    full_index_list = raw_indices * 4

    def parse_index(idx_str):
        """Splits '150a' into (' 150', 'a') for PDB format alignment."""
        match = re.match(r"(\d+)([a-zA-Z]?)", idx_str)
        if match:
            num = match.group(1)
            icode = match.group(2) if match.group(2) else " "
            return f"{num:>4}", icode
        return f"{idx_str:>4}", " "

    # 3. Process the PDB
    new_pdb_lines = []
    residue_count = -1
    last_res_id = None

    with open(pdb_input, 'r') as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                # PDB fixed-width columns:
                # Residue Name: 17-20
                # Chain ID: 21
                # Res Sequence Number: 22-26
                # Insertion Code: 26 (0-indexed: 26)

                res_name = line[17:20].strip()
                chain_id = line[21]
                res_seq = line[22:26].strip()
                icode = line[26]

                # Create a unique ID for the current residue in the input file
                current_res_id = (chain_id, res_seq, icode)

                # If we hit a new residue, move to the next index in our Excel list
                if current_res_id != last_res_id:
                    residue_count += 1
                    last_res_id = current_res_id

                # Get the new index from our expanded list
                if residue_count < len(full_index_list):
                    new_num, new_icode = parse_index(full_index_list[residue_count])

                    # Construct the new line with updated numbering
                    # Columns 22-26: Sequence number (4 chars)
                    # Column 26 (0-indexed): Insertion code
                    line = line[:22] + new_num + new_icode + line[27:]

                new_pdb_lines.append(line)
            else:
                # Keep TER, END, CRYST1, etc. as they are
                new_pdb_lines.append(line)

    # 4. Write the output
    with open(pdb_output, 'w') as f:
        f.writelines(new_pdb_lines)

    print(f"Success! Processed {residue_count + 1} residues.")
    print(f"Output saved to: {pdb_output}")

# Run the function
renumber_pdb("./data/Y221H.pdb", "~/Downloads/pdc3_numbering.xlsx", "MT_renumber.pdb")

In [ ]:
def get_clean_mapping_dict(pdb_path, split_features):
    from Bio import PDB
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('protein', pdb_path)

    # 1. Build the index-to-residue list
    pdb_residues = []
    for model in structure:
        for chain in model:
            c_id = chain.get_id().strip()
            prefix = f"{c_id}:" if c_id else ""
            for residue in chain:
                # FIX: Check if it's a chain residue (not water/ligand)
                # residue.get_id()[0] is the hetero-flag. ' ' means it's an ATOM record.
                if residue.get_id()[0] == ' ':
                    res_num = residue.get_id()[1]
                    pdb_residues.append(f"{prefix}{res_num}")

    # 2. Create the dictionary
    mapping_dict = {}
    for feat in split_features:
        try:
            parts = feat.replace(':', '-').split('-')
            idx1, idx2, interaction = int(parts[0]), int(parts[1]), parts[2]

            new_name = f"{pdb_residues[idx1]}-{pdb_residues[idx2]} ({interaction})"
            mapping_dict[feat] = new_name
        except Exception as e:
            mapping_dict[feat] = feat

    return mapping_dict

# --- Workflow ---

# 1. Generate the mapping
rename_map = get_clean_mapping_dict("./WT_renumber.pdb", split_features)
rename_map

### Step 3: Validate the Model

#### Interaction Occupation
Note that the 'X' is the modified iput after applying hystersis. For accurate look up, please refer to the second dataframe calculated by 'X_raw'. The first dataframe serves as a sanity check to see whether the KinTree is splitting based on stable transitions rather than artificial noise or transient "flickering."

In [ ]:
presence_df = macro_feature_presence(
    WT_hysteresis,
    feature_names_WT,
    leaf_WT,          # from collapse_labels after spectral lumping
    set(split_features)    # features used by your tree
)
presence_df = presence_df.rename(columns=rename_map)
display(presence_df)

In [ ]:
presence_df = macro_feature_presence(
    X_WT,
    feature_names_WT,
    leaf_WT,          # from collapse_labels after spectral lumping
    set(split_features)    # features used by your tree
)
presence_df = presence_df.rename(columns=rename_map)
display(presence_df)

In [ ]:
presence_df = macro_feature_presence(
    X_MT,
    feature_names_MT,
    leaf_MT,          # from collapse_labels after spectral lumping
    set(split_features)    # features used by your tree
)
presence_df = presence_df.rename(columns=rename_map)
display(presence_df)

### Step 4: Visualization
Here we provide Visualization functions to facilitate usage of our model. This includes:
1. Flux between macrostates
2. Three styles tree visualization
3. A function to extract representative frames from the trajectory for futher visualization
4. A ready to use pml (pymol input) file to directly open in pymol for high quality visualization output.
5. An interactive interface to visualize structure and the tree

#### Tree Visualization

In [ ]:
import plotly.express as px

with open('results/WT.json', 'r') as f:
    tree_data = json.load(f)

unique_macros = np.unique(leaf_WT)
colors_palette = px.colors.qualitative.Bold  # High contrast for papers
macro_color_map = {m: colors_palette[i % len(colors_palette)] for i, m in enumerate(unique_macros)}

In [ ]:
import json
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# 2. Setup Coloring based strictly on leaf_id
# Helper function to scan the JSON and find all unique leaf IDs
def get_all_leaf_ids(node):
    if node.get("leaf"):
        return [node.get("leaf_id")]
    leaves = []
    if "left" in node: leaves.extend(get_all_leaf_ids(node["left"]))
    if "right" in node: leaves.extend(get_all_leaf_ids(node["right"]))
    return leaves

unique_leaves = np.sort(np.unique(get_all_leaf_ids(tree_data)))

colors_palette = px.colors.qualitative.Bold  # High contrast for papers
# Map colors directly to the leaf_id
leaf_color_map = {l: colors_palette[i % len(colors_palette)] for i, l in enumerate(unique_leaves)}

# Clear lists
ids, labels, parents, values, colors = [], [], [], [], []

def parse_node(node, parent_id=None, connection="ROOT"):
    node_id = str(id(node))
    val = node.get("n_frames", 1)

    # Format the Connection Label
    path_info = ""
    if connection == "PRESENT":
        path_info = "<i>Yes:</i><br>"
    elif connection == "ABSENT":
        path_info = "<i>No:</i><br>"

    if not node.get("leaf"):
        raw_rule = node.get("rule", {}).get("name", "Split")
        rule_name = rename_map.get(raw_rule, raw_rule)
        display_label = f"{path_info}<b>{rule_name}?</b>"
        node_color = "rgb(245, 245, 245)"
    else:
        # We are completely ignoring macrostates here. Just using leaf_id.
        leaf_id = node.get("leaf_id")

        # 1-based indexing in the plot cells
        display_label = f"{path_info}<b>Leaf {leaf_id + 1}</b>"
        node_color = leaf_color_map[leaf_id]

    ids.append(node_id)
    labels.append(display_label)
    parents.append(parent_id)
    values.append(val)
    colors.append(node_color)

    if "left" in node:
        parse_node(node["left"], node_id, connection="PRESENT")
    if "right" in node:
        parse_node(node["right"], node_id, connection="ABSENT")

# Parse the tree
parse_node(tree_data)

# 3. Create Icicle Plot
fig = go.Figure()

fig.add_trace(go.Icicle(
    ids=ids,
    labels=labels,
    parents=parents,
    values=values,
    branchvalues="total",
    marker=dict(colors=colors, line=dict(color='white', width=1.5)),
    tiling=dict(orientation='v'),
    pathbar=dict(visible=True)
))

# 4. Final Styling
fig.update_layout(
    title=dict(
        text="Structural Path to Leaf Node Classification",
        y=0.95,
        x=0.05,
        xanchor='left'
    ),
    font=dict(size=20),
    template="plotly_white",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    showlegend=False,
    margin=dict(t=60, l=10, r=10, b=10),
)

# 5. Show the figure
fig.show(config={'displayModeBar': False})

In [ ]:
# --- Update the font sizes before saving ---
fig.update_layout(
    font=dict(size=22),  # Global font (Pathbar, etc.)
    title=dict(font=dict(size=30))  # Specific title size
)

# Update the labels inside the Icicle boxes
fig.update_traces(textfont_size=24)

# --- Now Save ---
fig.write_image("output/tree_publication_quality.pdf", width=1500, height=600)
fig.write_image("output/tree_high_res.png", width=1500, height=600, scale=3)

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# --- STEP 1: DEFINE COLORS BASED ON YOUR leaf_MT ---

# Ensure leaf_MT is a list of IDs you want to keep/color
# If leaf_MT is already defined in your environment, this uses it.
# We build the color map ONLY for IDs present in leaf_MT.
palette = px.colors.qualitative.Safe
macro_color_map = {l_id: palette[i % len(palette)] for i, l_id in enumerate(sorted(leaf_MT))}

# --- STEP 2: UPDATED PARSING FUNCTION ---

def parse_node(node, parent_id=None, connection="ROOT"):
    node_id = str(id(node))
    val = node.get("n_frames", 1)

    path_info = ""
    if connection == "PRESENT":
        path_info = "<i>Yes:</i><br>"
    elif connection == "ABSENT":
        path_info = "<i>No:</i><br>"

    if not node.get("leaf"):
        # Branch Nodes
        raw_rule = node.get("rule", {}).get("name", "Split")
        rule_name = rename_map.get(raw_rule, raw_rule) if 'rename_map' in globals() else raw_rule
        display_label = f"{path_info}<b>{rule_name}?</b>"
        node_color = "rgb(245, 245, 245)"
    else:
        # Leaf Nodes
        micro_id = node.get("leaf_id")

        # Check if this leaf exists in your leaf_MT
        if micro_id in leaf_MT:
            display_label = f"{path_info}<b>Leaf {micro_id + 1}</b>"
            node_color = macro_color_map[micro_id]
        else:
            # This handles the case where a leaf ID exists in the tree
            # but NOT in your leaf_MT list.
            display_label = f"{path_info}<b>New/Unknown Leaf {micro_id + 1}</b>"
            node_color = "rgb(200, 200, 200)" # Gray for non-matching leaves

    ids.append(node_id)
    labels.append(display_label)
    parents.append(parent_id)
    values.append(val)
    colors.append(node_color)

    if "left" in node:
        parse_node(node["left"], node_id, connection="PRESENT")
    if "right" in node:
        parse_node(node["right"], node_id, connection="ABSENT")

# --- STEP 3: EXECUTION ---

# Clear lists before re-parsing
ids, labels, parents, values, colors = [], [], [], [], []
parse_node(tree_data)

# --- STEP 4: CREATE ICICLE PLOT ---

fig = go.Figure()

fig.add_trace(go.Icicle(
    ids=ids,
    labels=labels,
    parents=parents,
    values=values,
    branchvalues="total",
    marker=dict(colors=colors, line=dict(color='white', width=1.5)),
    tiling=dict(orientation='v'),
    pathbar=dict(visible=True)
))

# --- STEP 5: LEGEND (ONLY SHOWING leaf_MT) ---

for m in sorted(leaf_MT):
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color=macro_color_map[m], symbol='square'),
        showlegend=True,
        name=f"Leaf {m+1}"
    ))

# --- STEP 6: STYLING ---

fig.update_layout(
    title=dict(
        text="Path Classification (Filtered by leaf_MT)",
        y=0.95,
        x=0.05,
        xanchor='left'
    ),
    template="plotly_white",
    margin=dict(t=60, l=10, r=10, b=10),
    showlegend=True
)

fig.show(config={'displayModeBar': False})

#### Extract frames for visualization

In [ ]:
# Extract frames for visualization
rng = np.random.default_rng(seed=42)  # set seed for reproducibility

macro_labels = np.asarray(leaf_WT)
unique_clusters = np.unique(macro_labels)

samples_per_cluster = 100
selected_frames = {}

for c in unique_clusters:
    frame_indices = np.where(macro_labels == c)[0]

    if len(frame_indices) < samples_per_cluster:
        raise ValueError(
            f"Cluster {c} has only {len(frame_indices)} frames, "
            f"cannot sample {samples_per_cluster}."
        )

    chosen = rng.choice(
        frame_indices,
        size=samples_per_cluster,
        replace=False
    )

    selected_frames[c] = np.sort(chosen)  # sorting is optional

In [ ]:
selected_frames

In [ ]:
trajectory = "./data/WT.xtc"
trajectory

In [ ]:
from MDAnalysis.coordinates.XTC import XTCWriter

u = mda.Universe('./WT_renumber.pdb', trajectory)
for cluster_id, frames in selected_frames.items():
    out_xtc = f"./output/WTmacro{cluster_id+1}_100frames.xtc"

    with XTCWriter(out_xtc, n_atoms=u.atoms.n_atoms) as W:
        for ts in u.trajectory[frames]:
            W.write(u.atoms)

    print(f"Wrote {out_xtc}")

#### Create PML file for pymol

In [ ]:
import re
from collections import defaultdict
import MDAnalysis as mda

FEATURE_RE = re.compile(r"^(?P<i>\d+)-(?P<j>\d+):(?P<itype>.+)$")

def _safe_token(x: str) -> str:
    return str(x).strip().replace(" ", "_").replace("-", "_").replace(":", "_")

def _resi_with_icode(res) -> str:
    """
    Return PyMOL-compatible resi string including insertion code if present.
    """
    resid = str(res.resid)
    icode = getattr(res, "icode", "")
    if icode and str(icode).strip() and str(icode).strip() != " ":
        return f"{resid}{str(icode).strip()}"
    return resid

def _infer_id_field(u: mda.Universe, prefer: str = "auto") -> str:
    """
    Decide whether to use PyMOL 'chain' or no identifier.

    Safer policy for ordinary PDB visualization:
    - prefer chain if chainIDs are present and meaningful
    - otherwise use plain resi
    - only use segi if explicitly requested
    """
    prefer = (prefer or "auto").lower()

    if prefer == "chain":
        return "chain"
    if prefer == "segi":
        return "segi"
    if prefer == "none":
        return "none"

    chain_vals = []
    for res in u.residues:
        chain = getattr(res, "chainID", "") or ""
        chain_vals.append(str(chain).strip())

    chain_nonempty = [c for c in chain_vals if c]

    if chain_nonempty:
        one_char_ratio = sum(len(c) == 1 for c in chain_nonempty) / len(chain_nonempty)
        if one_char_ratio >= 0.5:
            return "chain"

    return "none"

def _sel_term(object_name: str, id_field: str, id_value: str, resi: str) -> str:
    """
    Build a PyMOL selection term for one residue.
    """
    id_field = (id_field or "none").lower()
    id_value = str(id_value).strip()

    if id_field in ("chain", "segi") and id_value:
        return f"({object_name} and {id_field} {id_value} and resi {resi})"

    return f"({object_name} and resi {resi})"

def _pml_name_for_feature(i: int, j: int, itype: str) -> str:
    return f"sf_{i}_{j}_{_safe_token(itype)}"

def write_tree_split_pml(
    split_features,
    pdb_path,
    out_pml="tree_split_pairs.pml",
    out_tsv="tree_split_pairs.tsv",
    object_name="holo",

    # Robustness / overrides
    id_field="auto",           # "auto" (recommended), or "chain", or "segi"

    # Visualization options
    label_pairs=True,          # label residues by resn+resi (PDB)
    label_field="CA",          # typically "CA" for protein; if missing, set to None to label all atoms
    color_by_type=True,
    background="white",
    cartoon_transparency=0.15,
    stick_radius=0.18,
):
    """
    Always selects BOTH endpoints (i and j) for every feature.
    Works for holo (protein-ligand) and apo (protein-protein) without any special cases.
    """

    u = mda.Universe(pdb_path)
    id_field_used = _infer_id_field(u, prefer=id_field)

    # Parse features into rows and group by type
    rows = []
    rows_by_type = defaultdict(list)

    for feat in split_features:
        m = FEATURE_RE.match(feat)
        if not m:
            raise ValueError(f"Unrecognized feature format: {feat} (expected 'i-j:type')")

        i = int(m.group("i"))
        j = int(m.group("j"))
        itype = m.group("itype").strip()

        rows.append((feat, i, j, itype))
        rows_by_type[itype].append((feat, i, j, itype))

    # Map residue index -> (id_value, resi, resn)
    def idx_to_terms(idx: int):
        res = u.residues[idx]
        if id_field_used == "chain":
            id_value = getattr(res, "chainID", "") or ""
        elif id_field_used == "segi":
            id_value = getattr(res, "segid", "") or ""
        else:
            id_value = ""
        id_value = str(id_value).strip()
        resi = _resi_with_icode(res)
        resn = str(res.resname).strip()
        return id_value, resi, resn

    # Write TSV legend (one line per feature, with both endpoints)
    with open(out_tsv, "w") as f:
        f.write(
            "feature\tinteraction\t"
            "model_i\tmodel_j\t"
            f"{id_field_used}_i\tresi_i\tresn_i\t"
            f"{id_field_used}_j\tresi_j\tresn_j\n"
        )
        for feat, i, j, itype in rows:
            i_id, i_resi, i_resn = idx_to_terms(i)
            j_id, j_resi, j_resn = idx_to_terms(j)
            f.write(
                f"{feat}\t{itype}\t"
                f"{i}\t{j}\t"
                f"{i_id}\t{i_resi}\t{i_resn}\t"
                f"{j_id}\t{j_resi}\t{j_resn}\n"
            )

    # Simple palette for interaction types (fallback is yellow)
    type_color_map = {
        "hbond": "yellow",
        "pi_pi": "cyan",
        "cation_pi": "magenta",
        "t-shape": "cyan",
        "t_shape": "cyan",
        "salt_bridge": "orange",
        "disulfide": "green",
    }

    # Emit PML
    all_feature_sel_names = []

    with open(out_pml, "w") as p:
        p.write("# Auto-generated PyMOL script: Tree split-driving pairs\n")
        p.write("reinitialize\n")
        p.write(f'load "{pdb_path}", {object_name}\n\n')

        p.write("hide everything\n")
        p.write(f"show cartoon, {object_name}\n")
        p.write(f"color gray80, {object_name}\n")
        p.write(f"set cartoon_transparency, {cartoon_transparency}\n\n")

        p.write("# Debug helpers (uncomment if selections are empty)\n")
        p.write(f"# print cmd.get_chains('{object_name}')\n")
        p.write(f"# print cmd.get_segis('{object_name}')\n")
        p.write(f"# iterate ({object_name} and name CA), print(chain, segi, resn, resi)\n\n")

        p.write(f"# Using PyMOL id_field = {id_field_used} (user requested: {id_field})\n\n")

        p.write("set label_size, 18\n")
        p.write("set label_color, black\n\n")

        # Per-feature selections grouped by interaction type
        for itype in sorted(rows_by_type.keys()):
            group_name = f"splits_{_safe_token(itype)}"
            p.write(f"# === {itype} ===\n")

            for feat, i, j, itype in rows_by_type[itype]:
                i_id, i_resi, _ = idx_to_terms(i)
                j_id, j_resi, _ = idx_to_terms(j)

                term_i = _sel_term(object_name, id_field_used, i_id, i_resi)
                term_j = _sel_term(object_name, id_field_used, j_id, j_resi)

                sel_name = _pml_name_for_feature(i, j, itype)
                all_feature_sel_names.append(sel_name)

                # Pair selection: always both endpoints
                p.write(f"select {sel_name}, ({term_i}) or ({term_j})\n")
                p.write(f"show sticks, {sel_name}\n")
                p.write(f"group {group_name}, {sel_name}\n")

                # Labels: resn+resi (PDB-based). Usually CA for protein.
                if label_pairs:
                    if label_field:
                        p.write(f'label {sel_name} and name {label_field}, "%s%s" % (resn, resi)\n')
                    else:
                        p.write(f'label {sel_name}, "%s%s" % (resn, resi)\n')

                p.write("\n")

            # Color the whole group by interaction type
            if color_by_type:
                key = itype.strip()
                key2 = key.lower().replace(" ", "_")
                color = type_color_map.get(key, type_color_map.get(key2, "yellow"))
                p.write(f"color {color}, {group_name}\n\n")

            p.write("\n")

        # Combined selection
        if all_feature_sel_names:
            p.write(f"select split_all, {' or '.join(all_feature_sel_names)}\n")
        else:
            p.write("select split_all, none\n")

        p.write(f"set stick_radius, {stick_radius}, split_all\n")
        p.write("zoom split_all, 12\n")
        p.write(f"bg_color {background}\n")

    return {
        "out_pml": out_pml,
        "out_tsv": out_tsv,
        "id_field_used": id_field_used,
        "n_features": len(rows),
    }

In [ ]:
info = write_tree_split_pml(
    split_features=split_features,
    pdb_path='./WT_renumber.pdb',     # adjust if needed
    id_field="auto",           # or "segi" if your PDB uses segids like P0
    label_pairs=True,
    label_field="CA",          # if you want to label ligand too, set label_field=None (more clutter)
    color_by_type=True
)

print("Generated:", info["out_pml"], info["out_tsv"])
print("Using id_field:", info["id_field_used"])
print("Features:", info["n_features"])